# PhysAR-IDS -- NB8 (v6): ESI and PCR -- how stable and how channel-dependent are the explanations?

**What this measures.** For the tuned XGBoost ("standard") and the NB5b adversarially trained ("robust") model, exact TreeSHAP attributions on a stratified sample of the temporal test split (500 rows per class), on clean rows and on the same rows after three attacks, over 5 seeds:

* **PGD** (NB3 surrogate, 0.1 sigma on all 93 features): grey-box transfer, the same examples for both models.
* **White-box tree-greedy** (NB3c / NB5b code) at **0.3 and 1.0 sigma on the 60 cyber features**, run against the model being explained.

**ESI (Explanation Stability Index).** Per attack row, `L = ||phi(x_adv) - phi(x)||_2 / ||x_adv - x||_2` with `phi` the class-averaged |SHAP| vector (93 values), and `ESI = clip(1 - L, 0, 1)` as in the original definition. Because `ESI` saturates at 0 and depends on the perturbation size, the raw median `L`, the share of rows whose attribution does not change at all (no tree threshold crossed), the per-row **Spearman rank correlation** and the **top-10 overlap** of `phi` are reported next to it.

**Control (new, needed to read ESI at all).** Every attack is paired with a *random-feature control*: the same per-row perturbation magnitudes (same number of changed features, same sizes) reassigned to randomly chosen features among those the attack could touch, with random signs. A stability number is only evidence of adversarial instability if it is lower than the control's. Differences are tested with paired t-tests over the 5 seeds (Holm across the 6 model x attack cells).

**PCR (Physical-Channel Reliance; formerly "Physics Constrained Ratio").** Share of total attribution `sum |phi_i| / sum |phi_j|` carried by the 33 physical-channel features (11 raw sensor, 4 derived, 18 temporal); the original definition (the 4 derived features only) is reported as `PCR-derived`. Reported per class, standard vs robust, clean vs attacked, and for the rows the white-box attack actually evades. **Caveat to carry into the text:** the attacker only changes cyber features, so a shift of attribution away from them under attack is partly mechanical; the control separates that from an adversarial effect.

**Needs:** `phase1_data_v6.pkl`, `xgb_tuned_best_params_v6.json`, `surrogate_mlp_v6_seed{s}.pt` (NB3), `xgb_robust_v6_seed{s}.json` (NB5b). The standard model is refitted per seed with the NB2 tuned parameters (as in NB5 and NB-SHAP). **Runtime:** about 1 h on the local GPU.

## 0. Setup

In [ ]:
import os, sys, random, time, json, warnings
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from scipy.stats import rankdata
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
N_PER_CLASS = 500                                   # rows per class in the SHAP sample (benign included, for clean PCR)
PGD_EPS = 0.1; PGD_ITERS = 10                       # as NB3
WB_BUDGETS = {'0.3': 0.3, '1.0': 1.0}               # sigma, cyber features only
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
SHAP_CHUNK = 2048
LAYER1_ROB = 'xgb_robust_{v}_seed{s}.json'

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} backend: {BACKEND_NAME}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and feature groups

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']; atk_names = class_names[1:]
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
IS_CYBER = np.zeros(N_FEAT, bool); IS_CYBER[CYBER_IDX] = True; PHYS_IDX = np.where(~IS_CYBER)[0]
DERIVED = ['voltage_imbalance', 'current_imbalance', 'power_factor', 'freq_deviation']; DERIVED_IDX = np.array([FEATURE_COLS.index(f) for f in DERIVED])
assert len(PHYS_IDX) == 33 and IS_CYBER.sum() == 60 and not IS_CYBER[DERIVED_IDX].any()
rng0 = np.random.RandomState(0); npc = 120 if SMOKE_TEST else N_PER_CLASS
sel = np.concatenate([rng0.choice(np.where(y_test == c)[0], size=min(npc, int((y_test == c).sum())), replace=False) for c in range(6)])
X_s, y_s = X_test[sel], y_test[sel]; atk_s = y_s != 0
print(f'features {N_FEAT}: physical channel {len(PHYS_IDX)} (derived {len(DERIVED_IDX)}), cyber {int(IS_CYBER.sum())} | SHAP sample {len(sel)} rows, per class {np.bincount(y_s).tolist()} | test class counts {np.bincount(y_test).tolist()}')

## 2. White-box attack (NB3c / NB5b code) and PGD (NB3 code)

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def train_surrogate(seed, X_tr, y_tr, n_epochs=20):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed); torch.cuda.manual_seed_all(seed)
    model = SurrogateMLP(X_tr.shape[1]).to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long()); loader = DataLoader(ds, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed))
    for ep in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    model.eval(); return model
def pgd(model, X, y, eps=PGD_EPS, n_iter=PGD_ITERS, bs=65536):
    model.eval(); alpha = eps / 4; out = np.empty_like(X)
    for s in range(0, len(X), bs):
        yt = torch.from_numpy(y[s:s + bs]).long().to(DEVICE); X0 = torch.from_numpy(X[s:s + bs]).float().to(DEVICE); Xa = X0.clone()
        for _ in range(n_iter):
            Xa.requires_grad_(True); g = torch.autograd.grad(F.cross_entropy(model(Xa), yt), Xa)[0]; Xa = Xa.detach() + alpha * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
        out[s:s + bs] = Xa.detach().cpu().numpy()
    return out

def fit_pair(X, y, seed):
    """GPU-fitted classifier (used for the white-box margin search) and a CPU copy (the oracle for every reported number, as in NB4)."""
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X, y, verbose=False)
    tmp = f'_tmp_xgb_{seed}_{os.getpid()}.json'; clf.save_model(tmp); cpu = xgb.XGBClassifier(); cpu.load_model(tmp); cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp); return clf, cpu

def ppo_train_examples(seed, oracle):
    rng = np.random.RandomState(seed + 77); Xs, ys, used = [], [], []
    for tag in TRAIN_PPO_TAGS:
        pat, b = SAVED_AGENTS[tag]; path = pat.format(v=PIPELINE_VERSION, s=seed) + '.zip'
        if not os.path.exists(path): print(f'    {path} not found -- skipped'); continue
        idx = np.concatenate([rng.choice(np.flatnonzero(y_atk_train == c), min(N_PPO_TRAIN_PER_CLASS, int((y_atk_train == c).sum())), replace=False) for c in range(1, 6)])
        res = run_batched(RecurrentPPO.load(path), X_atk_train[idx], oracle, b, mode='ppo'); Xs.append(res['x_final']); ys.append(y_atk_train[idx]); used.append((tag, float(res['evaded'].mean())))
    return (np.vstack(Xs), np.concatenate(ys), used) if Xs else (np.zeros((0, N_FEAT), np.float32), np.zeros(0, np.int64), used)

def row_keys(X): return {r.tobytes() for r in np.ascontiguousarray(X, dtype=np.float32)}
def pgd_eval(cpu_m, X_adv):
    pred = cpu_m.predict(X_adv); ev = pred[atk_te] == 0
    return {'per_class_evasion': per_class(ev), 'overall_evasion': float(ev.mean()), 'weighted_f1': float(f1_score(y_test, pred, average='weighted')), 'benign_recall': float((pred[y_test == 0] == 0).mean()), 'attack_detected': float((~ev).mean())}
print('defence components defined')

## 3. TreeSHAP, ESI, PCR and the random-feature control

In [ ]:
def tree_shap(clf, X):
    """(n, 6, F+1) contributions in log-odds; last column = bias. Uses the fitted backend, falls back to CPU."""
    bst = clf.get_booster(); out = []
    for i in range(0, len(X), SHAP_CHUNK):
        dm = xgb.DMatrix(np.ascontiguousarray(X[i:i + SHAP_CHUNK]))
        try: out.append(bst.predict(dm, pred_contribs=True))
        except Exception: bst.set_param({'device': 'cpu'}); out.append(bst.predict(dm, pred_contribs=True))
    return np.concatenate(out)
def check_additivity(clf, X, phi, tol=1e-3):
    margin = clf.get_booster().predict(xgb.DMatrix(np.ascontiguousarray(X)), output_margin=True); err = float(np.abs(phi.sum(2) - margin).max()); assert err < tol, f'additivity violated: {err}'; return err
def phi_of(clf, X): return np.abs(tree_shap(clf, X)[:, :, :-1]).mean(1)          # class-averaged |SHAP|, (n, F)

def esi_parts(pc, pa, Xc, Xa):
    dphi = np.linalg.norm(pa - pc, axis=1); dx = np.linalg.norm(Xa - Xc, axis=1); ok = dx > 1e-8
    L = np.full(len(dx), np.nan); L[ok] = dphi[ok] / dx[ok]; esi = np.where(ok, np.clip(1 - np.nan_to_num(L), 0, 1), 1.0); return L, esi, ok, dphi
def spearman_rows(a, b):
    ra = rankdata(a, axis=1); rb = rankdata(b, axis=1); ra = ra - ra.mean(1, keepdims=True); rb = rb - rb.mean(1, keepdims=True)
    den = np.sqrt((ra ** 2).sum(1) * (rb ** 2).sum(1)); return np.where(den > 0, (ra * rb).sum(1) / np.where(den > 0, den, 1), 1.0)
def topk_overlap(a, b, k=10):
    ia = np.argsort(-a, 1)[:, :k]; ib = np.argsort(-b, 1)[:, :k]; return np.array([len(set(x) & set(y)) / k for x, y in zip(ia, ib)])
def pcr(phi, idx): tot = phi.sum(1); return np.where(tot > 1e-12, phi[:, idx].sum(1) / np.where(tot > 1e-12, tot, 1), 0.0)

def control(Xc, Xa, idx, rng):
    """random-feature control: the attack's per-row perturbation magnitudes reassigned to random features among `idx`, random signs."""
    d = (Xa - Xc)[:, idx]; perm = rng.rand(*d.shape).argsort(1); d = np.take_along_axis(d, perm, 1); sgn = rng.choice([-1.0, 1.0], size=d.shape)
    out = Xc.copy(); out[:, idx] = Xc[:, idx] + (np.abs(d) * sgn).astype(np.float32); return out

def summarise(pc, pa, Xc, Xa, y, pred_adv, pred_clean):
    L, esi, ok, dphi = esi_parts(pc, pa, Xc, Xa); rho = spearman_rows(pc, pa); top = topk_overlap(pc, pa); pr = pcr(pa, PHYS_IDX); prd = pcr(pa, DERIVED_IDX); pr0 = pcr(pc, PHYS_IDX)
    ev = pred_adv == 0; pcm = lambda v: np.array([v[y == c].mean() for c in range(1, 6)])
    return {'esi': float(esi.mean()), 'esi_pc': pcm(esi), 'L_median': float(np.nanmedian(L)), 'L_mean': float(np.nanmean(L)), 'unchanged': float((dphi[ok] == 0).mean()) if ok.any() else float('nan'), 'rho': float(rho.mean()), 'rho_pc': pcm(rho), 'top10': float(top.mean()),
            'pcr': float(pr.mean()), 'pcr_pc': pcm(pr), 'pcr_derived': float(prd.mean()), 'pcr_clean_same_rows': float(pr0.mean()), 'pcr_clean_pc': pcm(pr0), 'evasion': float(ev.mean()), 'evaded_n': int(ev.sum()),
            'pcr_evaded': float(pr[ev].mean()) if ev.any() else float('nan'), 'pcr_clean_evaded': float(pr0[ev].mean()) if ev.any() else float('nan'),
            'dx_l2': float(np.linalg.norm(Xa - Xc, axis=1).mean()), 'n_feat_changed': float((np.abs(Xa - Xc) > 1e-8).sum(1).mean())}
print('helpers defined')

## 4. Per-seed run

In [ ]:
RES = {}; t_all = time.time(); conds = ['pgd'] + [f'wb{b}' for b in WB_BUDGETS]
for seed in SEEDS:
    pf = f'esipcr_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = json.dumps([best_params, npc, seed, len(sel), float(X_s[:5].sum()), PGD_EPS, PGD_ITERS, WB_BUDGETS, K_CAND, MAX_ITERS, SMOKE_TEST, 'nb8-v1'], sort_keys=True, default=str)
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: RES[seed] = saved['res']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); res = {}; rng = np.random.RandomState(seed + 500)
    # models
    std_gpu = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); std_gpu.fit(X_train, y_train, verbose=False)
    tmp = f'_tmp_nb8_{seed}_{os.getpid()}.json'; std_gpu.save_model(tmp); std_cpu = xgb.XGBClassifier(); std_cpu.load_model(tmp); std_cpu.set_params(device='cpu', n_jobs=-1); os.remove(tmp)
    rob_gpu = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); rob_gpu.load_model(LAYER1_ROB.format(v=PIPELINE_VERSION, s=seed))
    rob_cpu = xgb.XGBClassifier(); rob_cpu.load_model(LAYER1_ROB.format(v=PIPELINE_VERSION, s=seed)); rob_cpu.set_params(device='cpu', n_jobs=-1)
    models = {'std': (std_gpu, std_cpu), 'rob': (rob_gpu, rob_cpu)}
    # PGD examples (NB3 surrogate, transfer) for the attack rows of the sample
    sur = SurrogateMLP(N_FEAT).to(DEVICE); sur.load_state_dict(torch.load(f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE)); sur.eval()
    Xa_atk = X_s[atk_s]; ya_atk = y_s[atk_s]; X_pgd = pgd(sur, Xa_atk, ya_atk)
    for mtag, (gpu_m, cpu_m) in models.items():
        phi_all = phi_of(gpu_m, X_s); r = {}
        add_err = check_additivity(gpu_m, X_s[:512], tree_shap(gpu_m, X_s[:512])) if seed == SEEDS[0] else None
        pred_clean = cpu_m.predict(X_s)
        r['clean'] = {'pcr_all_pc': np.array([pcr(phi_all[y_s == c], PHYS_IDX).mean() for c in range(6)]), 'pcr_derived_all_pc': np.array([pcr(phi_all[y_s == c], DERIVED_IDX).mean() for c in range(6)]), 'pcr': float(pcr(phi_all[atk_s], PHYS_IDX).mean()), 'pcr_derived': float(pcr(phi_all[atk_s], DERIVED_IDX).mean()),
                      'recall_pc': np.array([(pred_clean[y_s == c] == c).mean() for c in range(6)]), 'additivity_err': add_err}
        TH = get_thresholds(gpu_m); mg = Margin(gpu_m); phi_c = phi_all[atk_s]; pc_atk = pred_clean[atk_s]
        sets = {'pgd': (X_pgd, np.arange(N_FEAT))}
        for bt, B in WB_BUDGETS.items(): sets[f'wb{bt}'] = (whitebox_trace(mg, Xa_atk, ya_atk, CYBER_IDX, TH, B, verbose=False)[0], CYBER_IDX)
        for cname, (Xadv, idx) in sets.items():
            Xctl = control(Xa_atk, Xadv, idx, rng)
            r[cname] = summarise(phi_c, phi_of(gpu_m, Xadv), Xa_atk, Xadv, ya_atk, cpu_m.predict(Xadv), pc_atk)
            r[cname + '_ctrl'] = summarise(phi_c, phi_of(gpu_m, Xctl), Xa_atk, Xctl, ya_atk, cpu_m.predict(Xctl), pc_atk)
            a, c_ = r[cname], r[cname + '_ctrl']
            print(f'  [{mtag}] {cname:6s}: evasion {a["evasion"]:.3f} (control {c_["evasion"]:.3f}) | ESI {a["esi"]:.3f} vs control {c_["esi"]:.3f} | median L {a["L_median"]:.4f} vs {c_["L_median"]:.4f} | unchanged {a["unchanged"]:.3f} vs {c_["unchanged"]:.3f} | Spearman {a["rho"]:.3f} vs {c_["rho"]:.3f} | PCR {a["pcr_clean_same_rows"]:.3f} -> {a["pcr"]:.3f} (control {c_["pcr"]:.3f}) | changed features {a["n_feat_changed"]:.1f}', flush=True)
        res[mtag] = r; del mg
    RES[seed] = res; joblib.dump({'fp': cfg_fp, 'res': res}, pf); print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True); torch.cuda.empty_cache()
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. Results

In [ ]:
def ptest(a, b):
    d = np.asarray(a) - np.asarray(b)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (0.0 if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
ms = lambda x: f'{np.mean(x):.3f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.3f}'
V = lambda m, c, f: np.array([RES[s][m][c][f] for s in SEEDS]); VC = lambda m, c, f, i: np.array([RES[s][m][c][f][i] for s in SEEDS])
MODELS = {'std': 'standard (NB2 tuned)', 'rob': 'robust (NB5b)'}; ATK = {'pgd': 'PGD 0.1 sigma, all features (grey-box)', **{f'wb{b}': f'white-box {b} sigma, cyber features' for b in WB_BUDGETS}}
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30); T = {}
print('8a. Additivity check of exact TreeSHAP (seed %d, max |sum(phi) - margin|): std %.2e, rob %.2e' % (SEEDS[0], RES[SEEDS[0]]['std']['clean']['additivity_err'], RES[SEEDS[0]]['rob']['clean']['additivity_err']))
print('\n8b. Clean PCR (Physical-Channel Reliance = share of |SHAP| on the 33 physical-channel features), per class, mean +/- std over seeds; PCR-derived = the 4 derived features only')
rows = []
for m in MODELS: rows.append({'model': MODELS[m], **{c: ms([RES[s][m]['clean']['pcr_all_pc'][i] for s in SEEDS]) for i, c in enumerate(class_names)}})
for m in MODELS: rows.append({'model': MODELS[m] + ' [PCR-derived]', **{c: ms([RES[s][m]['clean']['pcr_derived_all_pc'][i] for s in SEEDS]) for i, c in enumerate(class_names)}})
T['pcr_clean'] = pd.DataFrame(rows); print(T['pcr_clean'].to_string(index=False))
pp = [ptest([RES[s]['std']['clean']['pcr_all_pc'][i] for s in SEEDS], [RES[s]['rob']['clean']['pcr_all_pc'][i] for s in SEEDS]) for i in range(6)]; ph = holm(pp)
print('   standard vs robust clean PCR, mean difference (robust - standard) and Holm p:', {c: (round(float(np.mean([RES[s]['rob']['clean']['pcr_all_pc'][i] - RES[s]['std']['clean']['pcr_all_pc'][i] for s in SEEDS])), 3), round(float(ph[i]), 4)) for i, c in enumerate(class_names)})
print('   clean per-class recall of the two models on this sample:', {c: (round(float(np.mean([RES[s]['std']['clean']['recall_pc'][i] for s in SEEDS])), 3), round(float(np.mean([RES[s]['rob']['clean']['recall_pc'][i] for s in SEEDS])), 3)) for i, c in enumerate(class_names)}, '(std, rob)')

print('\n8c. Explanation stability under attack vs the random-feature control (attack rows; paired t-test over seeds, Holm over the 6 model x attack cells per metric)')
cells_ = [(m, a) for m in MODELS for a in ATK]; p_esi = holm([ptest(V(m, a, 'esi'), V(m, a + '_ctrl', 'esi')) for m, a in cells_]); p_rho = holm([ptest(V(m, a, 'rho'), V(m, a + '_ctrl', 'rho')) for m, a in cells_]); p_L = holm([ptest(V(m, a, 'L_median'), V(m, a + '_ctrl', 'L_median')) for m, a in cells_])
rows = []
for k, (m, a) in enumerate(cells_):
    rows.append({'model': m, 'attack': a, 'evasion': ms(V(m, a, 'evasion')), 'evasion (control)': ms(V(m, a + '_ctrl', 'evasion')), 'features changed / row': f'{V(m, a, "n_feat_changed").mean():.1f}', 'ESI': ms(V(m, a, 'esi')), 'ESI (control)': ms(V(m, a + '_ctrl', 'esi')), 'p ESI': f'{p_esi[k]:.4f}',
                 'median L': f'{V(m, a, "L_median").mean():.4f}', 'median L (control)': f'{V(m, a + "_ctrl", "L_median").mean():.4f}', 'p L': f'{p_L[k]:.4f}', 'rows unchanged': ms(V(m, a, 'unchanged')), 'unchanged (control)': ms(V(m, a + '_ctrl', 'unchanged')),
                 'Spearman': ms(V(m, a, 'rho')), 'Spearman (control)': ms(V(m, a + '_ctrl', 'rho')), 'p rho': f'{p_rho[k]:.4f}', 'top-10 overlap': ms(V(m, a, 'top10')), 'top-10 (control)': ms(V(m, a + '_ctrl', 'top10'))})
T['stability'] = pd.DataFrame(rows); print(T['stability'].to_string(index=False))

print('\n8d. Per-class ESI under attack (mean over seeds)')
rows = []
for m in MODELS:
    for a in ATK: rows.append({'model': m, 'attack': a, **{c: f'{VC(m, a, "esi_pc", i).mean():.3f} (ctrl {VC(m, a + "_ctrl", "esi_pc", i).mean():.3f})' for i, c in enumerate(atk_names)}})
T['esi_per_class'] = pd.DataFrame(rows); print(T['esi_per_class'].to_string(index=False))
print('\n   standard vs robust, ESI under the same attack (paired over seeds, Holm over the 3 attacks):', {a: (round(float(V('std', a, 'esi').mean()), 3), round(float(V('rob', a, 'esi').mean()), 3)) for a in ATK}, 'p', dict(zip(ATK, np.round(holm([ptest(V('std', a, 'esi'), V('rob', a, 'esi')) for a in ATK]), 4))))

print('\n8e. PCR under attack (attack rows; clean PCR of the same rows -> attacked PCR; control = random-feature perturbation)')
p_pcr = holm([ptest(V(m, a, 'pcr'), V(m, a + '_ctrl', 'pcr')) for m, a in cells_]); rows = []
for k, (m, a) in enumerate(cells_):
    ev_n = V(m, a, 'evaded_n')
    rows.append({'model': m, 'attack': a, 'PCR clean': ms(V(m, a, 'pcr_clean_same_rows')), 'PCR attacked': ms(V(m, a, 'pcr')), 'PCR control': ms(V(m, a + '_ctrl', 'pcr')), 'change': f'{(V(m, a, "pcr") - V(m, a, "pcr_clean_same_rows")).mean():+.4f}', 'change (control)': f'{(V(m, a + "_ctrl", "pcr") - V(m, a, "pcr_clean_same_rows")).mean():+.4f}', 'p (attack vs control)': f'{p_pcr[k]:.4f}',
                 'PCR-derived attacked': ms(V(m, a, 'pcr_derived')), 'evaded rows (mean n)': f'{ev_n.mean():.0f}', 'PCR clean -> attacked, evaded rows only': (f'{np.nanmean(V(m, a, "pcr_clean_evaded")):.3f} -> {np.nanmean(V(m, a, "pcr_evaded")):.3f}' if np.isfinite(V(m, a, 'pcr_evaded')).any() else 'n/a')})
T['pcr_attack'] = pd.DataFrame(rows); print(T['pcr_attack'].to_string(index=False))
print('\n   per-class PCR clean -> attacked, standard model:')
for a in ATK: print(f'     {a:7s}', {c: f'{VC("std", a, "pcr_clean_pc", i).mean():.3f} -> {VC("std", a, "pcr_pc", i).mean():.3f}' for i, c in enumerate(atk_names)})

## 6. Figure and save

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(15, 10)); w = 0.2; xs = np.arange(len(ATK)); lab = list(ATK)
for off, (m, ctl) in enumerate([('std', False), ('std', True), ('rob', False), ('rob', True)]):
    k = lambda a: a + ('_ctrl' if ctl else '')
    ax[0, 0].bar(xs + (off - 1.5) * w, [V(m, k(a), 'esi').mean() for a in lab], w, yerr=[V(m, k(a), 'esi').std(ddof=1) for a in lab], capsize=2, label=f'{m}{" control" if ctl else ""}', alpha=0.45 if ctl else 0.9)
    ax[0, 1].bar(xs + (off - 1.5) * w, [V(m, k(a), 'rho').mean() for a in lab], w, yerr=[V(m, k(a), 'rho').std(ddof=1) for a in lab], capsize=2, label=f'{m}{" control" if ctl else ""}', alpha=0.45 if ctl else 0.9)
for a_, t_ in [(ax[0, 0], 'ESI under attack (higher = more stable)'), (ax[0, 1], 'Spearman rank correlation of the attribution vector')]:
    a_.set_xticks(xs); a_.set_xticklabels(lab, fontsize=8); a_.set_title(t_); a_.legend(fontsize=8); a_.grid(alpha=.3, axis='y'); a_.set_ylim(0, 1.05)
x6 = np.arange(6)
for off, m in enumerate(MODELS): ax[1, 0].bar(x6 + (off - 0.5) * 0.4, [np.mean([RES[s][m]['clean']['pcr_all_pc'][i] for s in SEEDS]) for i in range(6)], 0.4, yerr=[np.std([RES[s][m]['clean']['pcr_all_pc'][i] for s in SEEDS], ddof=1) for i in range(6)], capsize=2, label=MODELS[m])
ax[1, 0].set_xticks(x6); ax[1, 0].set_xticklabels(class_names, fontsize=8); ax[1, 0].set_ylabel('PCR (share of |SHAP| on the physical channel)'); ax[1, 0].set_title('Clean PCR per class'); ax[1, 0].legend(fontsize=8); ax[1, 0].grid(alpha=.3, axis='y')
x5 = np.arange(5); a = 'wb1.0'
for off, (lbl, f) in enumerate([('clean', lambda i: VC('std', a, 'pcr_clean_pc', i)), ('white-box 1.0 sigma', lambda i: VC('std', a, 'pcr_pc', i)), ('control', lambda i: VC('std', a + '_ctrl', 'pcr_pc', i))]):
    ax[1, 1].bar(x5 + (off - 1) * 0.27, [f(i).mean() for i in range(5)], 0.27, yerr=[f(i).std(ddof=1) for i in range(5)], capsize=2, label=lbl)
ax[1, 1].set_xticks(x5); ax[1, 1].set_xticklabels(atk_names, fontsize=8); ax[1, 1].set_title('PCR of the standard model: clean vs 1.0 sigma white-box vs random-feature control'); ax[1, 1].legend(fontsize=8); ax[1, 1].grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'esipcr_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': T, 'per_seed': {s: RES[s] for s in SEEDS}, 'config': {'N_PER_CLASS': npc, 'PGD_EPS': PGD_EPS, 'WB_BUDGETS': WB_BUDGETS, 'PHYS_IDX': PHYS_IDX, 'DERIVED': DERIVED}}, f'esipcr_results_{PIPELINE_VERSION}.pkl')
for k_, t_ in T.items(): t_.to_csv(f'esipcr_{k_}_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB8', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb8_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'esipcr_results_{PIPELINE_VERSION}.pkl', f'esipcr_{PIPELINE_VERSION}.png', f'nb8_run_manifest_{PIPELINE_VERSION}.json'] + [f'esipcr_{k_}_{PIPELINE_VERSION}.csv' for k_ in T]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)